<a href="https://colab.research.google.com/github/Mahiman001/Data-Science-learning-practice/blob/main/2SQL_ALONG_WITH_PYTHON2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas  as pd
import sqlite3

In [ ]:
conn = sqlite3.connect("memory:")

In [ ]:
df_rides = pd.read_csv('rides.csv')
df_stations = pd.read_csv('stations.csv')
df_users = pd.read_csv('users.csv')

In [ ]:
df_rides.columns

Index(['ride_id', 'user_id', 'start_station_id', 'end_station_id',
       'start_time', 'end_time', 'distance_km'],
      dtype='object')

In [ ]:
df_stations.columns

Index(['station_id', 'station_name', 'capacity', 'lat', 'lon'], dtype='object')

In [ ]:
df_users.columns

Index(['user_id', 'username', 'age', 'membership_level', 'created_at'], dtype='object')

In [ ]:
df_rides.to_sql('rides',conn,index = False, if_exists ='replace')
df_users.to_sql('users',conn,index = False, if_exists ='replace')
df_stations.to_sql('stations',conn,index = False, if_exists ='replace')

25

In [ ]:
query = """
SELECT
    (SELECT COUNT(*) FROM rides) AS total_rides,
    (SELECT COUNT(*) FROM stations) AS total_stations,
    (SELECT COUNT(*) FROM users) AS total_users;
"""

pd.read_sql(query, conn)

,total_rides,total_stations,total_users
0,15000,25,1000


In [ ]:
query = """
select r.ride_id,
r.distance_km,
u.username,
u.membership_level
from rides r inner join users u on r.user_id = u.user_id


"""
df_innerjoin = pd.read_sql(query,conn)

In [ ]:
df_innerjoin.head()

,ride_id,distance_km,username,membership_level
0,1,3.78,jennifer98,Casual
1,2,1.76,hbruce,Casual
2,3,2.83,andreabauer,Subscriber
3,4,7.64,owilkins,Casual
4,5,13.31,donald21,Casual


In [ ]:
df_rides.columns

Index(['ride_id', 'user_id', 'start_station_id', 'end_station_id',
       'start_time', 'end_time', 'distance_km'],
      dtype='object')

In [ ]:
query = """
select s.station_name,
     r.ride_id,
     r.distance_km,
     u.username
     from stations s
     left join rides r on s.station_id = r.start_station_id
     inner join users u on r.user_id = u.user_id


"""
df_start_station_info = pd.read_sql(query,conn)

In [ ]:
df_start_station_info.tail(20)

,station_name,ride_id,distance_km,username
14980,Amy Park St,14494,9.80,thomasrivers
14981,Amy Park St,14511,3.01,smora
14982,Amy Park St,14537,3.52,zbarr
14983,Amy Park St,14545,10.14,gary22
14984,Amy Park St,14553,6.17,wilcoxkayla
14985,Amy Park St,14624,8.55,steven71
14986,Amy Park St,14640,5.09,lopezrachel
14987,Amy Park St,14655,7.84,zacharyjohnson
14988,Amy Park St,14690,3.63,rubiomichael
14989,Amy Park St,14699,2.74,huntermark


In [ ]:
df_rides.head()

,ride_id,user_id,start_station_id,end_station_id,start_time,end_time,distance_km
0,1,542,17,5,2024-03-25 10:06:00,2024-03-25 10:24:55.074889,3.78
1,2,804,2,23,2024-02-05 16:53:00,2024-02-05 17:01:49.215435,1.76
2,3,328,13,23,2024-08-27 04:46:00,2024-08-27 05:00:10.059262,2.83
3,4,52,22,2,2024-01-03 00:39:00,2024-01-03 01:17:10.690811,7.64
4,5,449,23,10,2024-01-29 17:22:00,2024-01-29 18:28:32.193009,13.31


In [ ]:
df_rides.head()

,ride_id,user_id,start_station_id,end_station_id,start_time,end_time,distance_km
0,1,542,17,5,2024-03-25 10:06:00,2024-03-25 10:24:55.074889,3.78
1,2,804,2,23,2024-02-05 16:53:00,2024-02-05 17:01:49.215435,1.76
2,3,328,13,23,2024-08-27 04:46:00,2024-08-27 05:00:10.059262,2.83
3,4,52,22,2,2024-01-03 00:39:00,2024-01-03 01:17:10.690811,7.64
4,5,449,23,10,2024-01-29 17:22:00,2024-01-29 18:28:32.193009,13.31


In [ ]:
query = """
select
 u.username,
 start_station.station_name as start_station,
 end_station.station_name as end_station,
 r.distance_km
from rides r
inner join  users u
 on r.user_id = u.user_id
inner join stations start_station
 on r.start_station_id = start_station.station_id
inner join stations end_station
 on r.end_station_id = end_station.station_id
limit 10

"""
df_route_flow_info = pd.read_sql(query,conn)

In [ ]:
df_route_flow_info.head()

,username,start_station,end_station,distance_km
0,jennifer98,Edwards Drive St,Taylor Fall St,3.78
1,hbruce,Ryan Islands St,Michael Shores St,1.76
2,andreabauer,Stephanie Summit St,Michael Shores St,2.83
3,owilkins,Bell Villages St,Ryan Islands St,7.64
4,donald21,Michael Shores St,Brown Shoal St,13.31


In [ ]:
import pandas as pd

query = """
SELECT
    r.start_station_id,
    r.end_station_id,
    COALESCE(start_station.station_name, 'Unknown / Dockless') AS start_station,
    COALESCE(end_station.station_name, 'Unknown / Dockless') AS end_station,
    COUNT(r.ride_id) AS total_trips,
    ROUND(AVG(r.distance_km), 2) AS avg_distance_km,
    CASE
        WHEN r.start_station_id = r.end_station_id THEN 'Round Trip'
        ELSE 'Point-to-Point'
    END AS trip_type
FROM rides r
LEFT JOIN stations start_station
    ON r.start_station_id = start_station.station_id
LEFT JOIN stations end_station
    ON r.end_station_id = end_station.station_id
GROUP BY
    r.start_station_id,
    r.end_station_id,
    start_station.station_name,
    end_station.station_name,
    trip_type
ORDER BY
    total_trips DESC;
"""

df_route_flow_info_improved = pd.read_sql(query, conn)

In [ ]:
df_route_flow_info_improved.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625 entries, 0 to 624
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   start_station_id  625 non-null    int64  
 1   end_station_id    625 non-null    int64  
 2   start_station     625 non-null    object 
 3   end_station       625 non-null    object 
 4   total_trips       625 non-null    int64  
 5   avg_distance_km   625 non-null    float64
 6   trip_type         625 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 34.3+ KB


In [ ]:
df_rides.head()

,ride_id,user_id,start_station_id,end_station_id,start_time,end_time,distance_km
0,1,542,17,5,2024-03-25 10:06:00,2024-03-25 10:24:55.074889,3.78
1,2,804,2,23,2024-02-05 16:53:00,2024-02-05 17:01:49.215435,1.76
2,3,328,13,23,2024-08-27 04:46:00,2024-08-27 05:00:10.059262,2.83
3,4,52,22,2,2024-01-03 00:39:00,2024-01-03 01:17:10.690811,7.64
4,5,449,23,10,2024-01-29 17:22:00,2024-01-29 18:28:32.193009,13.31


In [ ]:
query = """
WITH user_rides_done as  (select count(*) as total_rides,user_id from rides  r
GROUP BY user_id)
select* from user_rides_done;


"""
df_user_rides_done = pd.read_sql(query,conn)

In [ ]:
df_user_rides_done.head(20)

,total_rides,user_id
0,15,1
1,19,2
2,16,3
3,12,4
4,14,5
5,20,6
6,14,7
7,13,8
8,16,9
9,11,10


In [ ]:
df_stations.columns

Index(['station_id', 'station_name', 'capacity', 'lat', 'lon'], dtype='object')

In [ ]:
query = """WITH station_avg AS (
    SELECT
        start_station_id,
        AVG(distance_km) AS avg_distance
    FROM rides
    GROUP BY start_station_id
)

SELECT
    s.station_name,
    sa.avg_distance
FROM station_avg sa
JOIN stations s
    ON sa.start_station_id = s.station_id
WHERE sa.avg_distance >
(
    SELECT AVG(distance_km)
    FROM rides
);
"""
df_major_stations = pd.read_sql(query,conn)

In [ ]:
df_major_stations.head(13)

,station_name,avg_distance
0,Megan Manors St,5.854196
1,Jimenez Summit St,5.854516
2,Richard Haven St,5.855508
3,Brown Shoal St,6.021721
4,Stephanie Summit St,5.975421
5,Samuel Manor St,6.016227
6,Smith Light St,5.912667
7,Christine Orchard St,5.895912
8,King Harbors St,5.985978
9,Christy Valley St,5.858605


In [ ]:
df_stations.columns

Index(['station_id', 'station_name', 'capacity', 'lat', 'lon'], dtype='object')

In [ ]:
query = """
select s.*,max(capacity) over() as max_caoacity
 from stations s;






"""
df_window = pd.read_sql(query,conn)


In [ ]:
df_users.columns

Index(['user_id', 'username', 'age', 'membership_level', 'created_at'], dtype='object')

In [ ]:
query = """
select r.ride_id,r.user_id,r.distance_km,
avg(r.distance_km)over(partition by r.user_id) as avg_distance,u.username
from rides r
join users u on r.user_id = u.user_id;

  """
df_avg = pd.read_sql(query,conn)

In [ ]:
df_rides.columns

Index(['ride_id', 'user_id', 'start_station_id', 'end_station_id',
       'start_time', 'end_time', 'distance_km'],
      dtype='object')

In [ ]:
query = """
select r.ride_id,
r.user_id,
r.distance_km,
u.username,
count(*)over(partition by r.user_id) as total_rides,
sum(distance_km)over(partition by r.user_id) as total_distance
from rides r
join users u on r.user_id = u.user_id;

"""
df_user_info = pd.read_sql(query,conn)


In [ ]:
df_user_info.head()

,ride_id,user_id,distance_km,username,total_rides,total_distance
0,188,1,3.86,enewman,15,118.11
1,773,1,12.16,enewman,15,118.11
2,1247,1,10.13,enewman,15,118.11
3,1514,1,10.61,enewman,15,118.11
4,2827,1,8.52,enewman,15,118.11


In [ ]:
query = """
select r.ride_id,
r.user_id,
r.distance_km,
u.username,
avg(r.distance_km)over(partition by r.user_id) as avg_dist,
case
      when distance_km > avg(r.distance_km)over(partition by r.user_id)
      then 'above average'
       else 'below average'
    end as ride_category
from rides r
join users u
on r.user_id = u.user_id;

"""
df_ride_type = pd.read_sql(query,conn)

In [ ]:
df_ride_type.head()

,ride_id,user_id,distance_km,username,avg_dist,ride_category
0,188,1,3.86,enewman,7.874,below average
1,773,1,12.16,enewman,7.874,above average
2,1247,1,10.13,enewman,7.874,above average
3,1514,1,10.61,enewman,7.874,above average
4,2827,1,8.52,enewman,7.874,above average


In [ ]:
query = """
with ride_stats as
(
select r.ride_id,
r.user_id,
r.distance_km,
u.username,
avg(r.distance_km)over(partition by r.user_id) as avg_dist
from rides r
join users u
on r.user_id = u.user_id

)

select *,
case when distance_km > avg_dist then 'above average '
else 'below average'
end as ride_category
from ride_stats;
"""
df_ride_stats = pd.read_sql(query,conn)

In [ ]:
df_ride_stats.head()

,ride_id,user_id,distance_km,username,avg_dist,ride_category
0,188,1,3.86,enewman,7.874,below average
1,773,1,12.16,enewman,7.874,above average
2,1247,1,10.13,enewman,7.874,above average
3,1514,1,10.61,enewman,7.874,above average
4,2827,1,8.52,enewman,7.874,above average


In [ ]:
df_rides.columns

Index(['ride_id', 'user_id', 'start_station_id', 'end_station_id',
       'start_time', 'end_time', 'distance_km'],
      dtype='object')

In [ ]:
query = """
with long_dist as (select r.user_id,
u.username,
r.ride_id,
r.distance_km,
row_number()over(partition by r.user_id order by r.distance_km DESC) as ridee_rank
from rides r
join users u on r.user_id = u.user_id
)

select * from long_dist
where  ridee_rank = 1;

"""
df_dist_info = pd.read_sql(query,conn)

In [ ]:
df_dist_info.head(20)

,user_id,username,ride_id,distance_km,ride_rank
0,1,enewman,9108,13.00,1
1,2,oliverdaniel,9174,12.69,1
2,3,pmartinez,2197,11.23,1
3,4,joseph98,5535,11.96,1
4,5,xstrong,7602,13.21,1
5,6,judy09,2865,9.86,1
6,7,yrobinson,3284,4.28,1
7,8,qwatson,10830,9.03,1
8,9,justinmejia,1760,11.99,1
9,10,mendozadavid,13870,12.81,1


In [ ]:
    query ="""

select r.user_id,
u.username,
r.ride_id,
r.distance_km,
rank()over(order by r.distance_km DESC) as ride_rank
from rides r
join users u
on r.user_id  =  u.user_id

"""
df_ab = pd.read_sql(query,conn)

In [ ]:
df_ab.head()

,user_id,username,ride_id,distance_km,ride_rank
0,968,jennifer97,13869,19.37,1
1,979,croy,1054,18.11,2
2,960,sharpbrandon,7269,17.98,3
3,778,benjaminshah,12186,17.86,4
4,280,staceydavis,5507,17.39,5


In [ ]:
query ="""

select r.user_id,
u.username,
r.ride_id,
r.distance_km,
dense_rank()over(order by r.distance_km DESC) as ride_rank
from rides r
join users u
on r.user_id  =  u.user_id

"""
df_ab = pd.read_sql(query,conn)